# 🌸 Nayari AI — Kaggle Export Notebook
> Use this notebook to export your trained Nayari AI model to different formats.

## 📦 Step 1 — Install

In [ ]:
%%capture
# Pin versions compatible with unsloth 2026.5.x
import os
os.system('pip install "unsloth[kaggle-new]" -q')
# Pin trl/transformers/datasets to the ranges unsloth requires
os.system(
    'pip install -q --upgrade '
    '"trl>=0.18.2,<=0.24.0" '
    '"transformers>=4.51.3,<=5.5.0" '
    '"datasets>=3.4.1,<4.4.0" '
    'accelerate peft bitsandbytes'
)


## 🔄 Step 8.5 — Reload Model for Export *(run only if skipping training)*
> Skip this cell if you just finished Step 7 (training) — `model` is already in memory.
>
> Run this cell **only** if you restarted the session or want to re-export without retraining.
> It auto-selects the best available checkpoint in order:
> 1. Merged 16-bit weights (`nayari_merged_16bit`) — fastest load
> 2. LoRA adapters (`nayari_lora`)
> 3. Latest training checkpoint (`nayari_checkpoints/checkpoint-*`)

In [ ]:
# ── Step 8.5: Reload model + tokenizer for export (skip if already in memory)
# ─────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048  # must match Step 4

MERGED_PATH      = "/kaggle/working/nayari_merged_16bit"
LORA_PATH        = "/kaggle/working/nayari_lora"
CHECKPOINT_ROOT  = "/kaggle/working/nayari_checkpoints"

def _already_loaded():
    """Return True if model/tokenizer are already in memory from training."""
    try:
        _ = model  # noqa: F821
        _ = tokenizer  # noqa: F821
        return True
    except NameError:
        return False

def _latest_checkpoint(root):
    """Return the path of the latest checkpoint-N folder, or None."""
    p = Path(root)
    if not p.exists():
        return None
    ckpts = sorted(
        [d for d in p.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    return str(ckpts[-1]) if ckpts else None

if _already_loaded():
    print("Model already in memory — skipping reload.")
else:
    # Priority 1: merged 16-bit (best for GGUF / push to HF)
    if Path(MERGED_PATH).exists():
        load_path = MERGED_PATH
        label = "merged 16-bit"

    # Priority 2: LoRA adapters
    elif Path(LORA_PATH).exists():
        load_path = LORA_PATH
        label = "LoRA adapters"

    # Priority 3: latest training checkpoint
    else:
        load_path = _latest_checkpoint(CHECKPOINT_ROOT)
        label = f"checkpoint ({Path(load_path).name})" if load_path else None

    if not load_path:
        raise FileNotFoundError(
            "No saved model found! Expected one of:\n"
            f"  {MERGED_PATH}\n"
            f"  {LORA_PATH}\n"
            f"  {CHECKPOINT_ROOT}/checkpoint-*/\n"
            "Run Step 7 (training) or Step 9A/9B first."
        )

    print(f"Loading from {label}: {load_path}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=load_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=False,  # must be False for correct GGUF export
    )
    print(f"Loaded from {label}")


## 💾 Step 9 — Save & Export
> **⚠️ Run this BEFORE Step 8 (Inference Test).**  
> `FastLanguageModel.for_inference()` in Step 8 patches the model away from PEFT state, which breaks all save methods below.

> **⚠️ Ensure Step 4 uses `load_in_4bit=False`** (already set).  
> With 4-bit quantisation the base weights cannot be dequantized, so the exported GGUF / merged model
> will contain only the generic base model — your fine-tuning is silently lost.

Run **one or more** of the blocks below. Each is independent.

| Block | Output | Size | Best for |
|---|---|---|---|
| **9A** | LoRA adapters | ~50 MB | Further training, merging later |
| **9B** | Merged 16-bit | ~3 GB | Full inference, HuggingFace upload |
| **9C** | GGUF Q4_K_M | ~1 GB | Ollama / LM Studio / llama.cpp |
| **9D** | GGUF Q8_0 | ~1.7 GB | Higher quality local inference |
| **9E** | HTTP Download | — | Tunnel download from Kaggle |
| **9F** | HuggingFace Hub | — | Permanent hosting at `Crossie/Nayari` |


### 9A — LoRA Adapters Only *(smallest, ~50 MB)*

In [ ]:
LORA_PATH = "/kaggle/working/nayari_lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)

import os
size_mb = sum(f.stat().st_size for f in Path(LORA_PATH).rglob("*") if f.is_file()) / 1024**2
print(f"✅ LoRA adapters saved → {LORA_PATH}")
print(f"   Size: {size_mb:.1f} MB")
print("   Load later with: model, tokenizer = FastLanguageModel.from_pretrained(LORA_PATH)")


### 9B — Merged 16-bit Model *(full weights, ~3 GB)*

In [ ]:
MERGED_PATH = "/kaggle/working/nayari_merged_16bit"

# save_method="merged_16bit" merges LoRA adapters into base model weights and
# saves as a standard HuggingFace model in bfloat16 precision.
model.save_pretrained_merged(
    MERGED_PATH, tokenizer,
    save_method="merged_16bit",
)

size_gb = sum(f.stat().st_size for f in Path(MERGED_PATH).rglob("*") if f.is_file()) / 1024**3
print(f"✅ Merged 16-bit saved → {MERGED_PATH}")
print(f"   Size: {size_gb:.2f} GB  (includes full merged weights — this is the fine-tuned model)")
print("   Use with: AutoModelForCausalLM.from_pretrained(MERGED_PATH)")


### 9C — GGUF Q4_K_M *(quantised, ~1 GB — Ollama / LM Studio)*

In [ ]:
GGUF_Q4_PATH = "/kaggle/working/nayari_gguf_q4"

model.save_pretrained_gguf(
    GGUF_Q4_PATH, tokenizer,
    quantization_method="q4_k_m",
)

# Rename to clean nayari-Q4_K_M.gguf (handles duplicates safely)
for f in sorted(Path(GGUF_Q4_PATH).rglob("*.gguf")):
    target = f.parent / "nayari-Q4_K_M.gguf"
    if f.name == target.name:
        print(f"Already named: {f} ({f.stat().st_size/1024**3:.2f} GB)")
        continue
    if target.exists():
        target.unlink()
    f.rename(target)
    print(f"GGUF Q4_K_M -> {target} ({target.stat().st_size/1024**3:.2f} GB)")
    break  # only rename the first one
print("   Load in Ollama: ollama create nayari -f Modelfile")
print("   Load in LM Studio: import the .gguf file directly")


### 9D — GGUF Q8_0 *(higher quality, ~1.7 GB)*

In [ ]:
# 9D reloads from the already-merged 16-bit weights saved in 9B.
# This avoids the transformers revert_weight_conversion bug that fires
# when save_pretrained_gguf is called on a live PEFT / for_inference model.
# Run 9B before this cell.
GGUF_Q8_PATH = "/kaggle/working/nayari_gguf_q8"
MERGED_PATH  = "/kaggle/working/nayari_merged_16bit"

from unsloth import FastLanguageModel as _FLM
model_q8, tok_q8 = _FLM.from_pretrained(
    model_name=MERGED_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)
model_q8.save_pretrained_gguf(
    GGUF_Q8_PATH, tok_q8,
    quantization_method="q8_0",
)

# Rename to clean nayari-Q8_0.gguf
for f in sorted(Path(GGUF_Q8_PATH).rglob("*.gguf")):
    new_name = f.parent / "nayari-Q8_0.gguf"
    f.rename(new_name)
    print(f"✅ GGUF Q8_0 → {new_name}  ({new_name.stat().st_size/1024**3:.2f} GB)")


### 9F — Push Everything to HuggingFace (`Crossie/Nayari`)
Pushes **merged 16-bit weights + GGUF Q4_K_M + GGUF Q8_0** in one go.

> **Before running:** Add your HuggingFace Write token to Kaggle Secrets:
> Kaggle → **Add-ons → Secrets** → **Add Secret** → Name: `HF_TOKEN`
> ([Get token](https://huggingface.co/settings/tokens) — needs **Write** scope)

> Run **Step 9B** (merged 16-bit save) before this cell.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import login, HfApi

REPO_ID    = "Crossie/Nayari"
IS_PRIVATE = False

# =====================================================================
#  UPLOAD MODE  -  pick one:
#    "weights_only"  =  merged 16-bit weights only (fastest upload)
#    "everything"    =  weights + Q4_K_M GGUF + Q8_0 GGUF
# =====================================================================
UPLOAD_MODE = "weights_only"   # <-- change to "everything" for full upload

# -- Auth -----------------------------------------------------------------
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as _e:
    print(f"Kaggle Secrets unavailable: {_e}")

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN", "").strip()
    if hf_token:
        print("HF_TOKEN loaded from environment variable")

if not hf_token:
    raise RuntimeError("HF_TOKEN not found! Add it in Kaggle Secrets or as env var.")

login(token=hf_token, add_to_git_credential=False)
print("Logged in to HuggingFace")
print(f"Upload mode: {UPLOAD_MODE}\n")

api = HfApi(token=hf_token)
api.create_repo(repo_id=REPO_ID, private=IS_PRIVATE, exist_ok=True)
print(f"Repo ready: https://huggingface.co/{REPO_ID}")

def safe_rename_gguf(directory, target_name):
    """Rename the first .gguf in directory to target_name. Handles all edge cases."""
    gguf_files = sorted(Path(directory).rglob("*.gguf"))
    if not gguf_files:
        return []
    src = gguf_files[0]
    dst = src.parent / target_name
    if src == dst or src.name == target_name:
        return [src]
    if dst.exists():
        dst.unlink()
    src.rename(dst)
    return [dst]

# -- 1. Upload merged 16-bit weights (always) -----------------------------
MERGED_PATH = "/kaggle/working/nayari_merged_16bit"
if Path(MERGED_PATH).exists():
    print(f"\nUploading merged 16-bit model ...")
    api.upload_folder(
        folder_path=MERGED_PATH,
        repo_id=REPO_ID,
        commit_message="Add merged 16-bit model",
    )
    print("Merged 16-bit uploaded")
else:
    print(f"WARNING: {MERGED_PATH} not found - run Step 9B first.")

if UPLOAD_MODE == "everything":
    # -- 2. GGUF Q4_K_M: generate if missing, then upload -----------------
    GGUF_Q4_PATH = "/kaggle/working/nayari_gguf_q4"
    q4_files = list(Path(GGUF_Q4_PATH).rglob("*.gguf")) if Path(GGUF_Q4_PATH).exists() else []

    if not q4_files:
        print("\nNo Q4_K_M GGUF found locally - generating now ...")
        model.save_pretrained_gguf(
            GGUF_Q4_PATH, tokenizer,
            quantization_method="q4_k_m",
        )
        print("Q4_K_M GGUF generated")

    q4_files = safe_rename_gguf(GGUF_Q4_PATH, "nayari-Q4_K_M.gguf")
    if q4_files:
        print("Uploading GGUF Q4_K_M ...")
        for f in q4_files:
            api.upload_file(
                path_or_fileobj=str(f),
                path_in_repo=f.name,
                repo_id=REPO_ID,
                commit_message=f"Add {f.name}",
            )
            print(f"  Uploaded {f.name} ({f.stat().st_size/1024**3:.2f} GB)")

    # -- 3. GGUF Q8_0: generate if missing, then upload -------------------
    GGUF_Q8_PATH = "/kaggle/working/nayari_gguf_q8"
    q8_files = list(Path(GGUF_Q8_PATH).rglob("*.gguf")) if Path(GGUF_Q8_PATH).exists() else []

    if not q8_files:
        print("\nNo Q8_0 GGUF found locally - generating now ...")
        from unsloth import FastLanguageModel as _FLM
        _m_q8, _t_q8 = _FLM.from_pretrained(
            model_name=MERGED_PATH,
            max_seq_length=2048,
            dtype=None,
            load_in_4bit=False,
        )
        _m_q8.save_pretrained_gguf(
            GGUF_Q8_PATH, _t_q8,
            quantization_method="q8_0",
        )
        del _m_q8, _t_q8
        print("Q8_0 GGUF generated")

    q8_files = safe_rename_gguf(GGUF_Q8_PATH, "nayari-Q8_0.gguf")
    if q8_files:
        print("Uploading GGUF Q8_0 ...")
        for f in q8_files:
            api.upload_file(
                path_or_fileobj=str(f),
                path_in_repo=f.name,
                repo_id=REPO_ID,
                commit_message=f"Add {f.name}",
            )
            print(f"  Uploaded {f.name} ({f.stat().st_size/1024**3:.2f} GB)")
else:
    print("\nSkipping GGUF uploads (mode = weights_only)")
    print("Change UPLOAD_MODE to 'everything' to include GGUFs.")

print(f"\nAll done! https://huggingface.co/{REPO_ID}")


# 9E — HTTP Download Links for ALL GGUF Files
Finds every .gguf file in /kaggle/working, starts a Cloudflare tunnel and prints a direct download URL.
No account needed. Links valid while session is open.

In [ ]:
import subprocess, time, socket, re, os, urllib.request
from pathlib import Path
from IPython.display import display, HTML

PORT        = 8989
SERVE_DIR   = "/kaggle/working"
SEARCH_ROOT = Path(SERVE_DIR)
CF_BIN      = "/kaggle/working/cloudflared"

# -- Check internet -------------------------------------------------------
def internet_ok():
    try:
        socket.setdefaulttimeout(5)
        socket.create_connection(("8.8.8.8", 53))
        return True
    except OSError:
        return False

if not internet_ok():
    print("No internet. Enable it in Kaggle Settings, then restart session.")
    raise SystemExit
print("Internet OK")

# -- Find GGUF files ------------------------------------------------------
gguf_files = sorted(SEARCH_ROOT.rglob("*.gguf"))
if not gguf_files:
    print("No .gguf files found. Run Step 9C or 9D first.")
    raise SystemExit

print(f"Found {len(gguf_files)} GGUF file(s):")
for g in gguf_files:
    print(f"   {g.relative_to(SEARCH_ROOT)}  ({g.stat().st_size/1024**3:.2f} GB)")
print()

# -- Kill any old server/tunnel processes ----------------------------------
subprocess.run(["pkill", "-f", f"http.server.*{PORT}"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

# -- Download cloudflared if needed ----------------------------------------
if not Path(CF_BIN).exists():
    print("Downloading cloudflared ...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", CF_BIN
    ], check=True)
    os.chmod(CF_BIN, 0o755)
    print("cloudflared ready")
else:
    print("cloudflared already present")
    os.chmod(CF_BIN, 0o755)

# -- Start HTTP server and verify it's alive -------------------------------
print(f"\nStarting HTTP server on port {PORT} ...")
server_proc = subprocess.Popen(
    ["python", "-m", "http.server", str(PORT), "--directory", SERVE_DIR,
     "--bind", "0.0.0.0"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Wait and verify the server is actually responding
server_alive = False
for attempt in range(10):
    time.sleep(1)
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/", timeout=3)
        server_alive = True
        break
    except Exception:
        pass

if not server_alive:
    print(f"HTTP server failed to start on port {PORT}!")
    print("Try restarting the Kaggle session.")
    server_proc.terminate()
    raise SystemExit

print(f"HTTP server running and verified (PID {server_proc.pid})")

# -- Start Cloudflare tunnel -----------------------------------------------
print("\nStarting Cloudflare tunnel (may take 15-30s) ...")
cf_log  = "/kaggle/working/cf_tunnel.log"
# Remove old log to avoid matching stale URLs
if Path(cf_log).exists():
    Path(cf_log).unlink()

cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
     "--no-autoupdate", "--logfile", cf_log,
     "--protocol", "http2"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

tunnel_url = None
for _ in range(60):  # wait up to 60 seconds
    time.sleep(1)
    if Path(cf_log).exists():
        log_text = Path(cf_log).read_text(errors="ignore")
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', log_text)
        if m:
            tunnel_url = m.group(0)
            # Verify the tunnel is actually working
            time.sleep(3)  # give it a moment to stabilize
            try:
                resp = urllib.request.urlopen(tunnel_url, timeout=10)
                if resp.status == 200:
                    break
            except Exception:
                # Tunnel URL found but not yet stable, keep waiting
                pass

if not tunnel_url:
    print("Tunnel URL not found after 60s.")
    if Path(cf_log).exists():
        print("Last 500 chars of tunnel log:")
        print(Path(cf_log).read_text(errors="ignore")[-500:])
    if 'server_proc' in globals(): server_proc.terminate()
    if 'cf_proc' in globals(): cf_proc.terminate()
    raise SystemExit

print(f"Tunnel live: {tunnel_url}\n")

# -- Print download links --------------------------------------------------
rows = ""
print("-- Download links --")
for g in gguf_files:
    rel     = g.relative_to(SEARCH_ROOT)
    size_gb = g.stat().st_size / 1024**3
    url     = f"{tunnel_url}/{rel}"
    print(f"  {g.name}  ({size_gb:.2f} GB)")
    print(f"     {url}")
    print(f'     wget "{url}" -O "{g.name}"')
    print()
    rows += (f"<tr><td><b>{g.name}</b></td><td>{size_gb:.2f} GB</td>"
             f"<td><a href='{url}' target='_blank'>Download</a></td>"
             f'<td><code>wget "{url}" -O "{g.name}"</code></td></tr>')

display(HTML(f"""
<h3>GGUF Download Links (Cloudflare Tunnel)</h3>
<p style='color:orange'>Links are active only while this session is open.</p>
<table border='1' cellpadding='6' cellspacing='0'
       style='border-collapse:collapse;font-family:monospace;font-size:13px'>
  <tr style='background:#f0f0f0'><th>File</th><th>Size</th><th>Link</th><th>wget command</th></tr>
  {rows}
</table>
<p>Browse all: <a href='{tunnel_url}' target='_blank'>{tunnel_url}</a></p>
"""))
print("Keep this notebook session OPEN while downloading.")
print("Run the stop cell below when done.")


# 9G-STOP — Run this to shut down the tunnel + server

In [ ]:
if 'server_proc' in globals():
    server_proc.terminate()
if 'cf_proc' in globals():
    cf_proc.terminate()
print("✅ Server and tunnel stopped.")
